In [4]:
from data.dataset import MalwareDatasetLoader
import pandas as pd

loader = MalwareDatasetLoader()
df_train, df_val, df_test = loader.make_data_splits()

print("Train shape:", df_train.shape)
print("Val shape:", df_val.shape)
print("Test shape:", df_test.shape)

df_train.head()


/Users/dian/STAT-841-cr-an-di/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using file: /Users/dian/.cache/kagglehub/datasets/agungpambudi/network-malware-detection-connection-analysis/versions/3/CTU-IoT-Malware-Capture-35-1conn.log.labeled.csv
Using file: /Users/dian/.cache/kagglehub/datasets/agungpambudi/network-malware-detection-connection-analysis/versions/3/CTU-IoT-Malware-Capture-3-1conn.log.labeled.csv
Using file: /Users/dian/.cache/kagglehub/datasets/agungpambudi/network-malware-detection-connection-analysis/versions/3/CTU-IoT-Malware-Capture-9-1conn.log.labeled.csv
Using file: /Users/dian/.cache/kagglehub/datasets/agungpambudi/network-malware-detection-connection-analysis/versions/3/CTU-IoT-Malware-Capture-1-1conn.log.labeled.csv
Using file: /Users/dian/.cache/kagglehub/datasets/agungpambudi/network-malware-detection-connection-analysis/versions/3/CTU-IoT-Malware-Capture-21-1conn.log.labeled.csv
Using file: /Users/dian/.cache/kagglehub/datasets/agungpambudi/network-malware-detection-connection-analysis/versions/3/CTU-IoT-Malware-Capture-34-1conn.log.l

/Users/dian/STAT-841-cr-an-di/data/dataset.py:28: DtypeWarning: Columns (8,9,10) have mixed types. Specify dtype option on import or set low_memory=False.
  dataframes.append(pd.read_csv(full_path, sep="|"))


Using file: /Users/dian/.cache/kagglehub/datasets/agungpambudi/network-malware-detection-connection-analysis/versions/3/CTU-IoT-Malware-Capture-42-1conn.log.labeled.csv


/Users/dian/STAT-841-cr-an-di/data/dataset.py:31: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df = df.replace("-", np.nan)


Train: 17507702
Val: 3751650
Test: 3751651
Train shape: (17507702, 23)
Val shape: (3751650, 23)
Test shape: (3751651, 23)


,ts,uid,id.orig_h,id.orig_p,id.resp_h,id.resp_p,proto,service,duration,orig_bytes,...,local_resp,missed_bytes,history,orig_pkts,orig_ip_bytes,resp_pkts,resp_ip_bytes,tunnel_parents,label,detailed-label
24898157,1.551412e+09,C6b59J3VbHsL3op3q6,192.168.1.200,35760.0,164.221.163.185,23.0,tcp,NaN,3.094754,0,...,NaN,0.0,S,6.0,360.0,0.0,0.0,NaN,Malicious PartOfAHorizontalPortScan,NaN
9976480,1.545484e+09,CN9TaEIG6bx8adaMh,192.168.1.196,60560.0,105.114.83.120,23.0,tcp,NaN,NaN,NaN,...,NaN,0.0,S,1.0,60.0,0.0,0.0,NaN,Benign,NaN
20635291,1.569018e+09,C4qdS929q3oML6ZH5g,192.168.1.195,629.0,162.248.88.215,62336.0,tcp,NaN,NaN,NaN,...,NaN,0.0,C,0.0,0.0,0.0,0.0,NaN,Malicious DDoS,NaN
14425760,1.532565e+09,CaB1AZf45diHhP2fh,192.168.100.111,31936.0,199.162.22.113,23.0,tcp,NaN,NaN,NaN,...,NaN,0.0,S,1.0,40.0,0.0,0.0,NaN,Malicious,PartOfAHorizontalPortScan
23992740,1.551402e+09,C1JTJL1n39lq4BQht4,192.168.1.200,56486.0,192.149.223.91,23.0,tcp,NaN,0.000002,0,...,NaN,0.0,S,2.0,120.0,0.0,0.0,NaN,Malicious PartOfAHorizontalPortScan,NaN


In [5]:
# use 10% data for test
df_train_small = df_train.sample(frac=0.1, random_state=42)
df_val_small = df_val.sample(frac=0.1, random_state=42)
df_test_small = df_test.sample(frac=0.1, random_state=42)

print(df_train_small.shape, df_val_small.shape, df_test_small.shape)


(1750770, 23) (375165, 23) (375165, 23)


In [6]:
import pandas as pd
import numpy as np

SKIP_COLS = [
    "ts",
    "uid",
    "id.orig_h",
    "id.resp_h",
    "tunnel_parents",
    "detailed-label",
    "local_orig",
    "local_resp",
]

CATEGORICAL_COLS = ["proto", "service", "conn_state", "history"]


def preprocess_binary_aligned(df_train, df_val, df_test):
    # Extract labels
    y_train = (df_train["label"] != "Benign").astype(int)
    y_val = (df_val["label"] != "Benign").astype(int)
    y_test = (df_test["label"] != "Benign").astype(int)
    
    # Drop unnecessary columns
    df_train = df_train.drop(columns=SKIP_COLS + ["label"])
    df_val = df_val.drop(columns=SKIP_COLS + ["label"])
    df_test = df_test.drop(columns=SKIP_COLS + ["label"])
    
    # Get dummies for ALL data together to ensure same columns
    df_all = pd.concat([df_train, df_val, df_test], keys=['train', 'val', 'test'])
    df_all = pd.get_dummies(df_all, columns=CATEGORICAL_COLS, dummy_na=True)
    
    # Convert any remaining object columns to numeric
    obj_cols = df_all.select_dtypes(include=["object"]).columns
    if len(obj_cols) > 0:
        df_all[obj_cols] = df_all[obj_cols].apply(pd.to_numeric, errors="coerce")
    
    # Split back
    X_train = df_all.xs('train')
    X_val = df_all.xs('val')
    X_test = df_all.xs('test')
    
    return X_train, y_train, X_val, y_val, X_test, y_test

X_train, y_train, X_val, y_val, X_test, y_test = preprocess_binary_aligned(
    df_train_small, df_val_small, df_test_small
)

print("Shapes:", X_train.shape, X_val.shape, X_test.shape)

Shapes: (1750770, 147) (375165, 147) (375165, 147)


In [7]:
import xgboost as xgb
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, confusion_matrix

def evaluate_binary(model, X, y, name=""):
    y_pred = model.predict(X)
    y_prob = model.predict_proba(X)[:, 1]

    acc = accuracy_score(y, y_pred)
    f1 = f1_score(y, y_pred)
    auc = roc_auc_score(y, y_prob)
    cm = confusion_matrix(y, y_pred)

    print(f"{name}")
    print(f"Accuracy: {acc:.4f}")
    print(f"F1:       {f1:.4f}")
    print(f"AUC:      {auc:.4f}")
    print("Confusion matrix:")
    print(cm)
    print()
    return acc, f1, auc, cm


xgb_clf = xgb.XGBClassifier(
    objective="binary:logistic",
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    tree_method="hist",
    eval_metric="logloss",
    random_state=42, 
)

xgb_clf.fit(X_train, y_train)

evaluate_binary(xgb_clf, X_train, y_train, name="Train")
evaluate_binary(xgb_clf, X_val, y_val, name="Val")
evaluate_binary(xgb_clf, X_test, y_test, name="Test")

Train
Accuracy: 0.9925
F1:       0.9942
AUC:      0.9974
Confusion matrix:
[[ 614804     227]
 [  12862 1122877]]

Val
Accuracy: 0.9924
F1:       0.9941
AUC:      0.9973
Confusion matrix:
[[131658     50]
 [  2798 240659]]

Test
Accuracy: 0.9924
F1:       0.9941
AUC:      0.9974
Confusion matrix:
[[131731     52]
 [  2785 240597]]



(0.992437993949329,
 0.9941388051591737,
 0.9973793351848386,
 array([[131731,     52],
        [  2785, 240597]]))

In [ ]:
import xgboost as xgb
import time

# Use full data 
X_train, y_train, X_val, y_val, X_test, y_test = preprocess_binary_aligned(
    df_train, df_val, df_test
)

xgb_clf = xgb.XGBClassifier(
    max_depth=5,              
    learning_rate=0.1,
    n_estimators=100,
    subsample=0.8,
    colsample_bytree=0.8,          
    objective='binary:logistic',
    tree_method='hist',
    random_state=42,
    n_jobs=-1
)

xgb_clf.fit(X_train, y_train)

evaluate_binary(xgb_clf, X_train, y_train, name="Train")
evaluate_binary(xgb_clf, X_val, y_val, name="Val")
test_results = evaluate_binary(xgb_clf, X_test, y_test, name="Test")

In [ ]:
import matplotlib.pyplot as plt

# Feature Importance 
importance = xgb_clf.feature_importances_
feature_names = X_train.columns
importance_df = pd.DataFrame({
    'feature': feature_names,
    'importance': importance
}).sort_values('importance', ascending=False).head(10)

plt.figure(figsize=(10, 6))
plt.barh(importance_df['feature'], importance_df['importance'])
plt.xlabel('Importance')
plt.title('Top 10 Feature Importance - XGBoost')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.savefig('xgboost_feature_importance.png')
plt.show()

In [ ]:
y_pred = xgb_clf.predict(X_test)
y_prob = xgb_clf.predict_proba(X_test)[:, 1]

# False Positives 
fp_mask = (y_pred == 1) & (y_test == 0)
fp_samples = X_test[fp_mask]
fp_probs = y_prob[fp_mask]

print(f"False Positives: {fp_mask.sum()}")
print(f"Average confidence: {fp_probs.mean():.4f}")

# False Negatives 
fn_mask = (y_pred == 0) & (y_test == 1)
fn_samples = X_test[fn_mask]
fn_probs = y_prob[fn_mask]

print(f"False Negatives: {fn_mask.sum()}")
print(f"Average confidence: {(1-fn_probs).mean():.4f}")

print("\nFalse Negative samples - Feature statistics:")
print(fn_samples.describe())

In [ ]:
import matplotlib.pyplot as plt

y_prob = xgb_clf.predict_proba(X_test)[:, 1]

plt.figure(figsize=(12, 5))

# Probability distribution of all samples
plt.subplot(1, 2, 1)
plt.hist(y_prob[y_test == 0], bins=50, alpha=0.5, label='Benign (actual)')
plt.hist(y_prob[y_test == 1], bins=50, alpha=0.5, label='Malicious (actual)')
plt.xlabel('Predicted Probability (Malicious)')
plt.ylabel('Count')
plt.title('Prediction Confidence Distribution')
plt.legend()

# Probability distribution of incorrect predictions
plt.subplot(1, 2, 2)
plt.hist(y_prob[fp_mask], bins=30, alpha=0.5, label='False Positives')
plt.hist(y_prob[fn_mask], bins=30, alpha=0.5, label='False Negatives')
plt.xlabel('Predicted Probability (Malicious)')
plt.ylabel('Count')
plt.title('Misclassified Samples')
plt.legend()

plt.tight_layout()
plt.savefig('prediction_confidence_analysis.png')
plt.show()

In [ ]:
import xgboost as xgb
from sklearn.model_selection import cross_val_score
import numpy as np
import time

print("XGBoost Hyperparameter Tuning")
print("\nStage 1: Tuning tree structure parameters")
print("Testing: max_depth and min_child_weight")
best_score = 0
best_params = {}

for max_depth in [4, 6, 8]:
    for min_child_weight in [1, 3, 5]:
        model = xgb.XGBClassifier(
            max_depth=max_depth,
            min_child_weight=min_child_weight,
            n_estimators=100,
            learning_rate=0.1,
            subsample=0.8,
            colsample_bytree=0.8,
            reg_lambda=1,  
            reg_alpha=0,   
            objective='binary:logistic',
            tree_method='hist',
            random_state=42,
            n_jobs=-1
        )
        
        scores = cross_val_score(model, X_train, y_train, cv=3, 
                                scoring='roc_auc', n_jobs=-1)
        mean_score = scores.mean()
        
        print(f"  max_depth={max_depth}, min_child_weight={min_child_weight}: "
              f"AUC = {mean_score:.4f}")
        
        if mean_score > best_score:
            best_score = mean_score
            best_params = {
                'max_depth': max_depth,
                'min_child_weight': min_child_weight
            }

print(f"\nBest Stage 1: {best_params}, AUC = {best_score:.4f}")

print("Stage 2: Tuning regularization (reg_lambda/ν)")
best_score_s2 = 0
best_lambda = 1

for reg_lambda in [0, 0.5, 1, 2, 5]:
    model = xgb.XGBClassifier(
        max_depth=best_params['max_depth'],
        min_child_weight=best_params['min_child_weight'],
        reg_lambda=reg_lambda, 
        reg_alpha=0,
        n_estimators=100,
        learning_rate=0.1,
        subsample=0.8,
        colsample_bytree=0.8,
        objective='binary:logistic',
        tree_method='hist',
        random_state=42,
        n_jobs=-1
    )
    
    scores = cross_val_score(model, X_train, y_train, cv=3, 
                            scoring='roc_auc', n_jobs=-1)
    mean_score = scores.mean()
    
    print(f"  reg_lambda={reg_lambda}: AUC = {mean_score:.4f}")
    
    if mean_score > best_score_s2:
        best_score_s2 = mean_score
        best_lambda = reg_lambda

best_params['reg_lambda'] = best_lambda
print(f"\nBest Stage 2: reg_lambda={best_lambda}, AUC = {best_score_s2:.4f}")


print("Stage 3: Tuning learning_rate and n_estimators")
best_score_s3 = 0
best_lr_params = {}

for learning_rate in [0.05, 0.1, 0.3]: 
    for n_estimators in [100, 200, 300]:
        model = xgb.XGBClassifier(
            max_depth=best_params['max_depth'],
            min_child_weight=best_params['min_child_weight'],
            reg_lambda=best_params['reg_lambda'],
            reg_alpha=0,
            learning_rate=learning_rate,
            n_estimators=n_estimators,
            subsample=0.8,
            colsample_bytree=0.8,
            objective='binary:logistic',
            tree_method='hist',
            random_state=42,
            n_jobs=-1
        )
        
        scores = cross_val_score(model, X_train, y_train, cv=3, 
                                scoring='roc_auc', n_jobs=-1)
        mean_score = scores.mean()
        
        print(f"  learning_rate={learning_rate}, n_estimators={n_estimators}: "
              f"AUC = {mean_score:.4f}")
        
        if mean_score > best_score_s3:
            best_score_s3 = mean_score
            best_lr_params = {
                'learning_rate': learning_rate,
                'n_estimators': n_estimators
            }

best_params.update(best_lr_params)
print(f"\nBest Stage 3: {best_lr_params}, AUC = {best_score_s3:.4f}")

# Final Model Training
print("Training Final Model")
print(f"Final parameters: {best_params}")

final_model = xgb.XGBClassifier(
    **best_params,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0,
    objective='binary:logistic',
    tree_method='hist',
    random_state=42,
    n_jobs=-1
)

start_time = time.time()
final_model.fit(X_train, y_train)
training_time = time.time() - start_time

print(f"\nTraining completed in {training_time/60:.2f} minutes")

# Evaluate
test_results = evaluate_binary(final_model, X_test, y_test, name="Final Model - Test")

# Save results
with open('xgboost_tuning_results.txt', 'w') as f:
    f.write("XGBoost Hyperparameter Tuning Results\n")
    f.write("=" * 50 + "\n\n")
    f.write(f"Best Parameters:\n{best_params}\n\n")
    f.write(f"Test Accuracy: {test_results[0]:.4f}\n")
    f.write(f"Test F1: {test_results[1]:.4f}\n")
    f.write(f"Test AUC: {test_results[2]:.4f}\n")
    f.write(f"Training time: {training_time/60:.2f} minutes\n")